In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# read data
import sys
sys.path.append('../..')
import data_preprocessing.coil20 as coil20
import data_preprocessing.cifar10 as cifar10
import data_preprocessing.reuters as reuters
import data_preprocessing.tdt2 as tdt2
from importlib import reload

# training process
import algorithm.algo2 as algo2
reload(algo2)

# evaluation
import algorithm.kmeans as kmeans
reload(kmeans)
import eval_metrics.ac, eval_metrics.nmi

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Using device]', device)

[Using device] cuda


In [ ]:
import argparse

parser = argparse.ArgumentParser(description='Train a model for clustering')
parser.add_argument('--n_clusters', type=int, default=20, help='number of clusters')
parser.add_argument('--loss', type=str, default='reconstruction_loss_without_regularization', help='loss function')
parser.add_argument('--lr', type=float, default=1e-4, help='learning rate')
parser.add_argument('--lamda', type=float, default=1, help='regularization parameter')
args = parser.parse_args()

DATASET = cifar10

CLUSTER_NUM = args.n_clusters
LOSS_FUNCTION_NAME = args.loss
LOSS_FUNCTION = getattr(origin, LOSS_FUNCTION_NAME)
LEARNING_RATE = args.lr
if LOSS_FUNCTION_NAME == 'reconstruction_loss_without_regularization':
    LAMBDA = None
else:
    LAMBDA = args.lamda

In [8]:
DATASET = coil20

CLUSTER_NUM = 10
LAMBDA = 1

# CLUSTER_NUM = 2, LAMBDA = 2, [4,20], 100 iter -> 100%
# [ 8 11] 80 iter -> 87.5%

# k=4, lambda=1, [ 5 10 12 17], 100 iter -> 94.4%
# k=6, lambda=1, [ 7 11 13 15 16 18], 100 iter -> 95.1%
# k=10, lambda=0.3, [ 2  4  5  7  8  9 15 17 18 20], 500 iter ->72.7%
# k=10, lambda=1, [ 1  2  4  6  8 11 12 13 15 16], 800 iter -> 0.81

In [15]:
# read data
mat, label = DATASET.read_data(num_of_class=CLUSTER_NUM, random_shuffle=True, 
                              random_choose_class=True)
mat = torch.tensor(mat, dtype=torch.float32)
mat -= torch.mean(mat)

print(f'[Using classes] {np.unique(np.array(label))}')

# compute the similarity matrix using cosine similarity
mat_normalized = mat / torch.norm(mat, dim=1).reshape(-1, 1)
A = mat_normalized @ mat_normalized.T
A.to(device)
pass

[Using classes] [ 1  2  4  6  8 10 11 12 14 18]


In [22]:
reload(algo2)
H = torch.randn(A.shape[0], CLUSTER_NUM)
H.to(device)

H_out = algo2.matrixFac(A=A, H=H, 
                        tol=1e-16,
                        lamda=10,
                        max_iter=10000000,
                        print_every=50)

In [23]:
reload(kmeans)
pred_labels, centroids = kmeans.kmeans_optimal(H_out.detach().cpu().numpy(), CLUSTER_NUM, max_iter=1000, turns=40)
acc = eval_metrics.ac.ac(label, pred_labels)
nmi = eval_metrics.nmi.nmi(label, pred_labels)
print(f'[Accuracy] {acc}\n[NMI] {nmi}')

[Accuracy] 0.39305555555555555
[NMI] 0.4894769213725838
